## 📚 目录
1. 环境配置与初始化
2. 核心组件介绍
3. 工具函数说明
4. 单轮检索示例
5. 多轮自主检索示例

### 1. 环境配置与初始化

首先导入必要的库并配置 FlashRAG 框架。

In [1]:
from flashrag.utils import get_retriever, get_generator
from flashrag.config import Config
import re
from typing import List, Dict

#### 1.1 配置参数说明

| 参数 | 说明 |
|------|------|
| `retrieval_method` | 检索方法，这里使用 e5 模型 |
| `model2path` | 模型路径映射 |
| `corpus_path` | 知识库语料路径 |
| `index_path` | FAISS 索引路径 |
| `retrieval_topk` | 每次检索返回的文档数量 |
| `generator_model_path` | 生成模型路径 |
| `faiss_gpu` | 是否使用 GPU 进行检索 |

FlashRAG支持更多参数调节，具体参考官方文档。

In [6]:
config_dict = {
    "retrieval_method": "e5",
    "model2path": {
        "e5": "/public/huggingface-models/intfloat/e5-base-v2",
    },
    "data_dir": "/root/FlashRAG/examples/quick_start/dataset/",
    "gpu_id": "0",
    "corpus_path": "/root/FlashRAG/examples/quick_start/indexes/general_knowledge.jsonl",
    "index_path": "/root/FlashRAG/examples/quick_start/indexes/e5_Flat.index",
    "faiss_gpu": True,
    "retrieval_topk": 5,
    #"generator_model_path": "/public/huggingface-models/Qwen/QwQ-32B",
    "generator_model_path": "/public/huggingface-models/meta-llama/Meta-Llama-3-8B",
    "gpu_memory_utilization": 0.8,
}

# 创建配置对象
config = Config("/root/FlashRAG/examples/methods/my_config.yaml", config_dict)

#### 1.2 初始化检索器和生成器

- **检索器 (Retriever)**: 负责从知识库中检索相关文档
- **生成器 (Generator)**: 负责基于检索到的文档生成回答

说明：demo中使用的检索器为了快速演示采用的是缩减版的语料库，实际需要使用完整版会由本平台提供，无需手动下载

In [7]:
print("正在初始化检索器和生成器...")
retriever = get_retriever(config)
generator = get_generator(config)
print("✓ 初始化完成！")

正在初始化检索器和生成器...
INFO 11-19 19:16:43 [utils.py:253] non-default args: {'max_model_len': 2048, 'gpu_memory_utilization': 0.8, 'max_logprobs': 32016, 'disable_log_stats': True, 'model': '/public/huggingface-models/meta-llama/Meta-Llama-3-8B'}
INFO 11-19 19:16:43 [model.py:631] Resolved architecture: LlamaForCausalLM
INFO 11-19 19:16:43 [model.py:1745] Using max model len 2048
INFO 11-19 19:16:43 [scheduler.py:216] Chunked prefill is enabled with max_num_batched_tokens=8192.
(EngineCore_DP0 pid=32871) INFO 11-19 19:16:49 [core.py:93] Initializing a V1 LLM engine (v0.11.1) with config: model='/public/huggingface-models/meta-llama/Meta-Llama-3-8B', speculative_config=None, tokenizer='/public/huggingface-models/meta-llama/Meta-Llama-3-8B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=2048, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_siz

Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  25% Completed | 1/4 [00:15<00:45, 15.15s/it]
Loading safetensors checkpoint shards:  50% Completed | 2/4 [00:29<00:29, 14.92s/it]
Loading safetensors checkpoint shards:  75% Completed | 3/4 [00:44<00:14, 14.88s/it]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:46<00:00,  9.85s/it]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:46<00:00, 11.72s/it]
(EngineCore_DP0 pid=32871) 


(EngineCore_DP0 pid=32871) INFO 11-19 19:17:37 [default_loader.py:314] Loading weights took 46.92 seconds
(EngineCore_DP0 pid=32871) INFO 11-19 19:17:38 [gpu_model_runner.py:3334] Model loading took 14.9596 GiB memory and 47.245298 seconds
(EngineCore_DP0 pid=32871) INFO 11-19 19:17:43 [backends.py:631] Using cache directory: /root/.cache/vllm/torch_compile_cache/e6acc7d9e4/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=32871) INFO 11-19 19:17:43 [backends.py:647] Dynamo bytecode transform time: 5.18 s
(EngineCore_DP0 pid=32871) INFO 11-19 19:17:47 [backends.py:251] Cache the graph for dynamic shape for later use
(EngineCore_DP0 pid=32871) INFO 11-19 19:17:55 [backends.py:282] Compiling a graph for dynamic shape takes 10.66 s
(EngineCore_DP0 pid=32871) INFO 11-19 19:17:56 [monitor.py:34] torch.compile takes 15.84 s in total
(EngineCore_DP0 pid=32871) INFO 11-19 19:17:57 [gpu_worker.py:359] Available KV cache memory: 2.58 GiB
(EngineCore_DP0 pid=32871) INFO 11-19 19:17:5

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:02<00:00, 20.58it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:01<00:00, 27.12it/s]


(EngineCore_DP0 pid=32871) INFO 11-19 19:18:02 [gpu_model_runner.py:4240] Graph capturing finished in 4 secs, took 0.50 GiB
(EngineCore_DP0 pid=32871) INFO 11-19 19:18:02 [core.py:250] init engine (profile, create kv cache, warmup model) took 24.27 seconds
INFO 11-19 19:18:03 [llm.py:352] Supported tasks: ['generate']
✓ 初始化完成！


### 2. 核心工具函数

以下是支持多轮检索问答系统的关键工具函数。

#### 2.1 文本提取函数

从生成的回复中提取特定标签之间的内容（如搜索查询、答案等）。

In [8]:
def extract_between(text: str, start_tag: str, end_tag: str):
    """
    提取文本中指定标签之间的内容
    
    参数:
        text: 原始文本
        start_tag: 起始标签
        end_tag: 结束标签
    
    返回:
        提取的内容（如果找到），否则返回 None
    
    示例:
        >>> text = "开始<query>人工智能</query>结束"
        >>> extract_between(text, "<query>", "</query>")
        '人工智能'
    """
    pattern = re.escape(start_tag) + r"(.*?)" + re.escape(end_tag)
    matches = re.findall(pattern, text, flags=re.DOTALL)
    if matches:
        return matches[-1].strip()  # 返回最后一个匹配项
    return None

In [9]:
# 测试提取函数
test_text = "这是一些文本 <|begin_search_query|>美国总统是谁<|end_search_query|> 更多文本"
extracted = extract_between(test_text, "<|begin_search_query|>", "<|end_search_query|>")
print(f"提取结果: {extracted}")

提取结果: 美国总统是谁


#### 2.2 文档格式化函数

将检索到的文档列表格式化为结构化字符串，方便模型理解。

In [10]:
def retrieved_docs_to_string(begin_doc_tag: str, end_doc_tag: str, retrieved_docs: List[Dict]):
    """
    将检索到的文档格式化为字符串
    
    参数:
        begin_doc_tag: 文档列表起始标签
        end_doc_tag: 文档列表结束标签
        retrieved_docs: 检索到的文档列表
    
    返回:
        格式化后的文档字符串
    
    格式示例:
        <|begin_search_result|>
        (1)Title: 文档标题1 Text: 文档内容1
        (2)Title: 文档标题2 Text: 文档内容2
        <|end_search_result|>
    """
    format_doc_string = ""
    for idx, doc in enumerate(retrieved_docs):
        contents = doc['contents']
        # 分离标题和正文
        title = contents.split('\n')[0]
        text = '\n'.join(contents.split('\n')[1:])
        doc_string = f"Title: {title} Text: {text}"
        # 移除开头的数字编号（如果有）
        doc_string = re.sub(r'^\d+\s+', '', doc_string)
        format_doc_string += f'({idx+1}){doc_string}\n'
    
    # 添加起始和结束标签
    format_doc_string = f'\n\n{begin_doc_tag}\n{format_doc_string}\n{end_doc_tag}\n\n'
    return format_doc_string

#### 2.3 系统提示词生成函数

生成指导模型进行多轮检索的系统提示词。

In [11]:
def get_multiqa_search_o1_instruction(MAX_SEARCH_LIMIT):
    """
    生成多轮检索问答系统的指令提示词
    
    参数:
        MAX_SEARCH_LIMIT: 最大搜索次数限制
    
    返回:
        系统提示词字符串
    """
    return (
        "You are a reasoning assistant with the ability to perform web searches to help "
        "you answer the user's question accurately. You have special tools:\n\n"
        "- To perform a search: write <|begin_search_query|> your query here <|end_search_query|>.\n"
        "Then, the system will search and analyze relevant web pages, then provide you with helpful information in the format <|begin_search_result|> ...search results... <|end_search_result|>.\n"
        "When you have gotten enough information to answer the user's question, stop searching and continue your reasoning to provide the your answer with <|begin_answer|> ... your answer... <|end_answer|>.\n\n"
        f"You can repeat the search process multiple times if necessary. The maximum number of search attempts is limited to {MAX_SEARCH_LIMIT}.\n\n"
        "Once you have all the information you need, continue your reasoning.\n\n"
        "Example:\n"
        "Question: \"Alice David is the voice of Lara Croft in a video game developed by which company?\"\n"
        "Assistant thinking steps:\n"
        "- I need to find out who voices Lara Croft in the video game.\n"
        "- Then, I need to determine which company developed that video game.\n\n"
        "Assistant:\n"
        "<|begin_search_query|>Alice David Lara Croft voice<|end_search_query|>\n\n"
        "(System returns processed information from relevant web pages)\n\n"
        "Assistant thinks: The search results indicate that Alice David is the voice of Lara Croft in a specific video game. Now, I need to find out which company developed that game.\n\n"
        "Assistant:\n"
        "<|begin_search_query|>video game developed by Alice David Lara Croft<|end_search_query|>\n\n"
        "(System returns processed information from relevant web pages)\n\n"
        "Assistant continues reasoning with the new information...\n\n"
        "Remember:\n"
        "- Use <|begin_search_query|> to request a web search and end with <|end_search_query|>.\n"
        "- When done searching, continue your reasoning.\n\n"
    )

In [12]:
# 配置参数
MAX_SEARCH_LIMIT = 3  # 最大搜索次数
stop_tokens = ["<|end_search_query|>", "<|end_answer|>"]  # 停止生成的标记

# 生成系统提示词
sysprompt = get_multiqa_search_o1_instruction(MAX_SEARCH_LIMIT)
print("系统提示词已生成")
print(f"前100个字符: {sysprompt[:100]}...")

系统提示词已生成
前100个字符: You are a reasoning assistant with the ability to perform web searches to help you answer the user's...


### 3. 基础功能演示

#### 3.1 批量检索示例

演示如何批量检索多个问题。

In [13]:
test_questions = [
    "Who is the president of the United States?",
    "What is the capital of France?",
    "Explain the theory of relativity.",
]

print("批量检索测试问题的相关文档...")
docs = retriever.batch_search(test_questions, 3)

for i, question in enumerate(test_questions):
    print(f"\n问题 {i+1}: {question}")
    print(f"检索到 {len(docs[i])} 个相关文档")
    if docs[i]:
        print(f"第一个文档标题: {docs[i][0]['contents'].split(chr(10))[0][:50]}...")

批量检索测试问题的相关文档...


Encoding process:   0%|          | 0/1 [00:00<?, ?it/s][2025-11-19 19:18:43] INFO langid.py:162: initializing identifier


Use `query: ` as retreival instruction


Encoding process: 100%|██████████| 1/1 [00:01<00:00,  1.92s/it]


问题 1: Who is the president of the United States?
检索到 3 个相关文档
第一个文档标题: What is the name of the current president of the U...

问题 2: What is the capital of France?
检索到 3 个相关文档
第一个文档标题: What is the capital of France??...

问题 3: Explain the theory of relativity.
检索到 3 个相关文档
第一个文档标题: "Research Einsteins theory of relativity and provi...


#### 3.2 简单问答（无检索）

首先测试不使用检索的直接问答。

In [14]:
print("\n" + "="*50)
print("简单问答测试（不使用检索）")
print("="*50 + "\n")


for i, question in enumerate(test_questions):
    print(f"问题 {i+1}: {question}\n")
    prompt = f"Answer the following question. Question: {question}\nAssistant thinking steps:\n"
    
    # 生成回答
    response = generator.generate(
        prompt,
        max_new_tokens=512,
        temperature=0.1,
        stop=stop_tokens,
    )
    
    print(f"回答:\n{response[0]}\n")
    print("-"*50 + "\n")


简单问答测试（不使用检索）

问题 1: Who is the president of the United States?



Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

回答:
1. The president of the United States is the head of state and head of government of the United States of America. The president directs the executive branch of the federal government and is the commander-in-chief of the United States Armed Forces.
2. The president is indirectly elected to a four-year term by the people through the Electoral College. The officeholder leads the nation in times of peace and serves as commander-in-chief in times of war.
3. The president is the only officeholder in the United States with the power to grant pardons and reprieves. The president also has the power to veto bills passed by Congress, but Congress may override these vetoes with a two-thirds vote of both houses.
4. The president is also responsible for appointing federal judges, ambassadors, and other executive branch officials. The president also has the power to issue executive orders, which have the force of law.
5. The president is also the head of the Democratic Party and the Republican P

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

回答:
1. The capital of France is Paris.
2. The capital of France is Paris.
3. The capital of France is Paris.
4. The capital of France is Paris.
5. The capital of France is Paris.
6. The capital of France is Paris.
7. The capital of France is Paris.
8. The capital of France is Paris.
9. The capital of France is Paris.
10. The capital of France is Paris.
11. The capital of France is Paris.
12. The capital of France is Paris.
13. The capital of France is Paris.
14. The capital of France is Paris.
15. The capital of France is Paris.
16. The capital of France is Paris.
17. The capital of France is Paris.
18. The capital of France is Paris.
19. The capital of France is Paris.
20. The capital of France is Paris.
21. The capital of France is Paris.
22. The capital of France is Paris.
23. The capital of France is Paris.
24. The capital of France is Paris.
25. The capital of France is Paris.
26. The capital of France is Paris.
27. The capital of France is Paris.
28. The capital of France is Pari

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

回答:
1. The theory of relativity is a theory that explains the relationship between space and time.
2. The theory of relativity was developed by Albert Einstein in 1905.
3. The theory of relativity states that the speed of light is constant in all reference frames.
4. The theory of relativity also states that time is relative and can be affected by motion.
5. The theory of relativity has been used to explain many phenomena, such as the bending of light by gravity and the expansion of the universe.
6. The theory of relativity is still being studied and is one of the most important theories in physics.
7. The theory of relativity has had a major impact on our understanding of the universe and has led to many new discoveries.
8. The theory of relativity is still being tested and is one of the most important theories in physics.
9. The theory of relativity has had a major impact on our understanding of the universe and has led to many new discoveries.
10. The theory of relativity is still b

### 4. 单轮检索完整流程

演示一个完整的单轮检索问答流程：
1. 模型生成搜索查询
2. 检索相关文档
3. 基于文档生成最终答案

In [15]:
print("\n" + "="*50)
print("单轮检索完整流程示例")
print("="*50 + "\n")

# 选择第一个问题进行演示
question = test_questions[0]
print(f"📝 问题: {question}\n")

# 步骤1: 生成搜索查询
print("步骤1: 让模型生成搜索查询...")
prompt = sysprompt + f"Question: {question}\n"
response = generator.generate(
    prompt, 
    max_new_tokens=512, 
    temperature=0.1, 
    stop=stop_tokens
)[0]
print(f"模型回复:\n{response}\n")

# 步骤2: 提取搜索查询
search_query = extract_between(response, "<|begin_search_query|>", "<|end_search_query|>")
print(f"🔍 提取的搜索查询: {search_query}\n")

if search_query:
    # 步骤3: 执行检索
    print("步骤2: 执行检索...")
    docs = retriever.search(search_query, 3)
    print(f"检索到 {len(docs)} 个相关文档\n")
    
    # 步骤4: 格式化文档
    doc_str = retrieved_docs_to_string("<|begin_search_result|>", "<|end_search_result|>", docs)
    print(f"格式化文档（前200字符）:\n{doc_str[:200]}...\n")
    
    # 步骤5: 基于检索结果生成最终答案
    print("步骤3: 基于检索结果生成最终答案...")
    prompt = prompt + response + doc_str + '\n'
    response = generator.generate(
        prompt, 
        max_new_tokens=512, 
        temperature=0.1, 
        stop=stop_tokens
    )[0]
    print(f"模型回复:\n{response}\n")
    
    # 步骤6: 提取最终答案
    answer = extract_between(response, "<|begin_answer|>", "<|end_answer|>")
    if answer:
        print(f"✅ 最终答案:\n{answer}\n")
    else:
        print("⚠️ 未找到明确的答案标记\n")
else:
    print("⚠️ 模型未生成搜索查询\n")


单轮检索完整流程示例

📝 问题: Who is the president of the United States?

步骤1: 让模型生成搜索查询...


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

模型回复:
Assistant thinking steps:
- I need to find out who is the president of the United States.

Assistant:
<|begin_search_query|>president of the United States<|end_search_query|>

🔍 提取的搜索查询: president of the United States

步骤2: 执行检索...


Encoding process: 100%|██████████| 1/1 [00:00<00:00, 23.92it/s]

检索到 3 个相关文档

格式化文档（前200字符）:


<|begin_search_result|>
(1)Title: Who is the current President of the United States? Text: The current President of the United States is Joe Biden.\n
(2)Title: What is the name of the current presid...

步骤3: 基于检索结果生成最终答案...


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

模型回复:
Assistant thinks: The search results indicate that the current president of the United States is Joe Biden. Now, I need to find out more about him.

Assistant:
<|begin_search_query|>Joe Biden<|end_search_query|>

⚠️ 未找到明确的答案标记



### 5. 多轮自主检索系统

实现一个完整的多轮检索系统，模型可以自主决定是否需要继续搜索。

In [16]:
def multi_turn_qa(question: str, max_turns: int = 3, verbose: bool = True):
    """
    多轮检索问答函数
    
    参数:
        question: 用户问题
        max_turns: 最大检索轮数
        verbose: 是否打印详细信息
    
    返回:
        final_answer: 最终答案
        search_history: 搜索历史记录
    """
    if verbose:
        print(f"\n{'='*60}")
        print(f"开始多轮检索问答")
        print(f"{'='*60}\n")
        print(f"📝 问题: {question}\n")
    
    # 初始化
    sysprompt = get_multiqa_search_o1_instruction(max_turns)
    prompt = sysprompt + f"Question: {question}\n"
    search_history = []
    
    for turn in range(max_turns):
        if verbose:
            print(f"\n--- 第 {turn + 1} 轮 ---\n")
        
        # 生成回复
        response = generator.generate(
            prompt,
            max_new_tokens=512,
            temperature=0.1,
            stop=stop_tokens
        )[0]
        
        if verbose:
            print(f"模型回复:\n{response}\n")
        
        # 检查是否生成了搜索查询
        search_query = extract_between(response, "<|begin_search_query|>", "<|end_search_query|>")
        
        if search_query:
            if verbose:
                print(f"🔍 检测到搜索查询: {search_query}")
            
            # 执行检索
            docs = retriever.search(search_query, 3)
            search_history.append({
                'turn': turn + 1,
                'query': search_query,
                'num_docs': len(docs)
            })
            
            if verbose:
                print(f"✓ 检索到 {len(docs)} 个文档\n")
            
            # 格式化文档并添加到提示词
            doc_str = retrieved_docs_to_string("<|begin_search_result|>", "<|end_search_result|>", docs)
            prompt = prompt + response + doc_str + '\n'
            
        else:
            # 检查是否给出了最终答案
            answer = extract_between(response, "<|begin_answer|>", "<|end_answer|>")
            
            if answer:
                if verbose:
                    print(f"✅ 找到最终答案！\n")
                return answer, search_history
            else:
                if verbose:
                    print("⚠️ 未检测到搜索查询或最终答案，继续...\n")
                prompt = prompt + response + '\n'
    
    if verbose:
        print(f"\n⚠️ 达到最大检索轮数 ({max_turns})，尝试提取答案...\n")
    
    # 达到最大轮数，尝试提取答案
    final_response = generator.generate(
        prompt + "Please provide your final answer with <|begin_answer|> ... <|end_answer|>.\n",
        max_new_tokens=512,
        temperature=0.1,
        stop=stop_tokens
    )[0]
    
    answer = extract_between(final_response, "<|begin_answer|>", "<|end_answer|>")
    return answer if answer else "未能生成明确答案", search_history

#### 5.1 测试多轮检索系统

In [17]:
# 测试问题
complex_question = "Who is the president of the United States?"

# 执行多轮检索问答
final_answer, search_history = multi_turn_qa(
    complex_question, 
    max_turns=3, 
    verbose=True
)

# 输出结果摘要
print("\n" + "="*60)
print("结果摘要")
print("="*60 + "\n")
print(f"问题: {complex_question}\n")
print(f"搜索轮数: {len(search_history)}")
for search in search_history:
    print(f"  - 第{search['turn']}轮: 查询=\"{search['query']}\", 文档数={search['num_docs']}")
print(f"\n最终答案:\n{final_answer}\n")


开始多轮检索问答

📝 问题: Who is the president of the United States?


--- 第 1 轮 ---



Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

模型回复:
Assistant thinking steps:
- I need to find out who is the president of the United States.

Assistant:
<|begin_search_query|>president of the United States<|end_search_query|>

🔍 检测到搜索查询: president of the United States


Encoding process: 100%|██████████| 1/1 [00:00<00:00, 50.13it/s]

...

Encoding process: 100%|██████████| 1/1 [00:00<00:00, 60.49it/s]

✓ 检索到 3 个文档


⚠️ 达到最大检索轮数 (3)，尝试提取答案...



Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


结果摘要

问题: Who is the president of the United States?

搜索轮数: 3
  - 第1轮: 查询="president of the United States", 文档数=3
  - 第2轮: 查询="Joe Biden", 文档数=3
  - 第3轮: 查询="to request a web search and end with", 文档数=3

最终答案:
未能生成明确答案

